# RS-Code-SSM: DeepSeek-R1 Trace Generation on Kaggle T4

Generates verified reasoning traces using DeepSeek-R1-Distill-Qwen-7B on GPU.
Uploads results to HuggingFace Dataset when done.

## Setup
1. Enable GPU: **Settings → Accelerator → GPU T4 x2**
2. Enable Internet: **Settings → Internet → On** (needed for HuggingFace datasets + model)
3. Add secret: `HF_TOKEN` (HuggingFace write token)
4. Run all cells

**Expected runtime**: ~8–12 hours for 2000 verified traces (n_samples=4)

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'accelerate', 'bitsandbytes',
    'datasets', 'huggingface_hub', 'safetensors'
], check=True)
print('Dependencies installed')

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
import os

HF_TOKEN   = os.environ.get('HF_TOKEN', '')      # set in Kaggle Secrets
DATA_REPO  = 'pgalyen1987/rs-code-ssm-traces'    # ← your HF dataset repo
TEACHER    = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
OUTPUT     = 'reasoning_traces_r1.jsonl'

N_SAMPLES  = 4       # candidates per problem (rejection sampling)
MAX_TOKENS = 2048    # max tokens per generation
TEMPERATURE = 0.7
TOP_P      = 0.95
EXEC_TIMEOUT = 10    # seconds for test execution

print(f'Teacher: {TEACHER}')
print(f'Output:  {OUTPUT}')
print(f'HF repo: {DATA_REPO}')

In [ ]:
# ── Cell 3: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

In [ ]:
# ── Cell 4: Load teacher model ────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print(f'Loading {TEACHER}...')
tokenizer = AutoTokenizer.from_pretrained(TEACHER, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    TEACHER,
    torch_dtype=torch.bfloat16,
    device_map='auto',       # spreads across both T4s if available
    trust_remote_code=True,
)
model.eval()
print('Model loaded.')
print(f'Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')

In [ ]:
# ── Cell 5: Generation + execution helpers ─────────────────────────────────────
import re, json, subprocess, tempfile, os, textwrap

SYSTEM_PROMPT = """You are an expert Python programmer. Think step by step before writing code.
Use <think> tags to show your reasoning, then provide the solution.
Format:
<think>
[your chain-of-thought reasoning]
</think>
```python
[your solution]
```"""

def build_prompt(problem_prompt: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{problem_prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def parse_response(text: str):
    think_m = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    thinking = think_m.group(1).strip() if think_m else ''
    code_m = re.search(r'```python\s*(.*?)```', text, re.DOTALL)
    if not code_m:
        code_m = re.search(r'```\s*(.*?)```', text, re.DOTALL)
    solution = code_m.group(1).strip() if code_m else ''
    return thinking, solution

@torch.inference_mode()
def generate(prompt: str, temperature: float = TEMPERATURE) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    input_len = inputs['input_ids'].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        temperature=temperature,
        top_p=TOP_P,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def execute_solution(code: str, test_code: str, timeout: int = EXEC_TIMEOUT) -> bool:
    if not code.strip():
        return False
    combined = code + '\n\n' + test_code
    with tempfile.NamedTemporaryFile(suffix='.py', mode='w', delete=False) as f:
        f.write(combined)
        fname = f.name
    try:
        result = subprocess.run(
            [sys.executable, fname],
            capture_output=True, timeout=timeout
        )
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    except Exception:
        return False
    finally:
        os.unlink(fname)

print('Helpers ready.')

In [ ]:
# ── Cell 6: Load problems (HumanEval + MBPP) ──────────────────────────────────
from datasets import load_dataset

problems = []

# HumanEval
he = load_dataset('openai_humaneval', split='test')
for row in he:
    problems.append({
        'problem_id': row['task_id'],
        'prompt': row['prompt'],
        'test_code': row['test'] + f"\ncheck({row['entry_point']})",
        'source': 'humaneval',
    })
print(f'HumanEval: {len([p for p in problems if p["source"]=="humaneval"])} problems')

# MBPP (full config: test_setup_code for imports; sanitized: test_imports)
mbpp = load_dataset('mbpp', split='test')
for row in mbpp:
    setup = row.get('test_setup_code', '') or '\n'.join(row.get('test_imports', []))
    tests = '\n'.join(row.get('test_list', []))
    test_code = (setup + '\n\n' + tests) if setup.strip() else tests
    problems.append({
        'problem_id': f'MBPP/{row["task_id"]}',
        'prompt': row.get('text', row.get('prompt', '')),
        'test_code': test_code,
        'source': 'mbpp',
    })
print(f'MBPP: {len([p for p in problems if p["source"]=="mbpp"])} problems')
print(f'Total: {len(problems)} problems')

In [ ]:
# ── Cell 7: Resume from existing output ───────────────────────────────────────
from pathlib import Path

done_ids = set()
if Path(OUTPUT).exists():
    with open(OUTPUT) as f:
        for line in f:
            try:
                done_ids.add(json.loads(line)['problem_id'])
            except Exception:
                pass
    print(f'Resuming: {len(done_ids)} problems already done')
else:
    print('Starting fresh')

remaining = [p for p in problems if p['problem_id'] not in done_ids]
print(f'{len(remaining)} problems to generate')

In [ ]:
# ── Cell 8: Generate traces with rejection sampling ────────────────────────────
import time

n_verified = len(done_ids)
n_failed = 0
start_time = time.time()

with open(OUTPUT, 'a') as out_f:
    for i, problem in enumerate(remaining):
        pid = problem['problem_id']
        prompt_text = build_prompt(problem['prompt'])
        test_code = problem.get('test_code', '')

        best_thinking = ''
        best_solution = ''
        passed = False

        for attempt in range(N_SAMPLES):
            # Increase temperature slightly for diversity
            temp = TEMPERATURE + 0.1 * attempt
            try:
                raw = generate(prompt_text, temperature=min(temp, 1.0))
            except Exception as e:
                print(f'  [{pid}] generation error: {e}')
                continue

            thinking, solution = parse_response(raw)
            if not solution:
                continue

            # Keep longest thinking as fallback
            if len(thinking) > len(best_thinking):
                best_thinking = thinking
                best_solution = solution

            if test_code and execute_solution(solution, test_code):
                best_thinking = thinking
                best_solution = solution
                passed = True
                break

        if not best_solution:
            n_failed += 1
            print(f'[{i+1}/{len(remaining)}] SKIP {pid} — no solution extracted')
            continue

        if test_code and not passed:
            # verified_only: skip problems where no sample passed
            n_failed += 1
            print(f'[{i+1}/{len(remaining)}] FAIL {pid} — no sample passed tests')
            continue

        trace = {
            'problem_id': pid,
            'source': problem['source'],
            'prompt': problem['prompt'],
            'thinking': best_thinking,
            'solution': best_solution,
            'passed_tests': passed,
            'n_samples': N_SAMPLES,
        }
        out_f.write(json.dumps(trace) + '\n')
        out_f.flush()
        n_verified += 1

        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed * 3600
        print(f'[{i+1}/{len(remaining)}] OK {pid} | verified={n_verified} failed={n_failed} | {rate:.0f} prob/hr')

print(f'\nDone. {n_verified} verified traces, {n_failed} failed.')

In [ ]:
# ── Cell 9: Upload to HuggingFace Dataset ─────────────────────────────────────
from huggingface_hub import HfApi
from pathlib import Path

n_lines = sum(1 for _ in open(OUTPUT))
print(f'Uploading {OUTPUT} ({n_lines} traces) to {DATA_REPO}...')

if HF_TOKEN and DATA_REPO:
    api = HfApi(token=HF_TOKEN)

    # Create repo if it doesn't exist
    try:
        api.create_repo(DATA_REPO, repo_type='dataset', exist_ok=True)
    except Exception as e:
        print(f'Repo create: {e}')

    api.upload_file(
        path_or_fileobj=OUTPUT,
        path_in_repo='reasoning_traces_r1.jsonl',
        repo_id=DATA_REPO,
        repo_type='dataset',
        token=HF_TOKEN,
    )
    print(f'Uploaded to https://huggingface.co/datasets/{DATA_REPO}')
else:
    print('Set HF_TOKEN in Kaggle Secrets to auto-upload.')
    print(f'Traces saved locally: {Path(OUTPUT).stat().st_size/1024:.0f} KB')